# Model Fine-Tuning

In [ ]:
# ============================================
# 1. UNSLOTH İNDİRME VE KURULUM
# ============================================
%%capture
!pip install --upgrade unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install unsloth_zoo

In [ ]:
# ============================================
# 2. TEMEL KÜTÜPHANELER
# ============================================
from unsloth import FastLanguageModel
import torch

import os

In [ ]:
# ============================================
# 3. MODEL YÜKLEME
# ============================================
max_seq_length = 1024
dtype = None  # Auto detection
load_in_4bit = True

# Desteklenen modeller
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/llama-3.2-3B-Instruct-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit"
]

In [ ]:
# ============================================
# 4. PRE-TRAINED MODEL YÜKLEME
# ============================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token="hf_..." , # Gerekirse HF token ekleyin
)

In [ ]:
# ============================================
# 5. BAŞLANGIÇ INFERENCE TESTİ (Opsiyonel)
# ============================================
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Kaza yaptım, sigorta şirketiyle nasıl ilerlemeliyim?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
print("\n=== EĞİTİM ÖNCESİ MODEL CEVABI ===")
_ = model.generate(input_ids, streamer=text_streamer, max_new_tokens=256,
                   pad_token_id=tokenizer.eos_token_id)

In [ ]:
# ============================================
# 6. LORA ADAPTERLARI EKLEME
# ============================================
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # LoRA rank - daha yüksek = daha fazla kapasite
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.15,  # Overfitting'i engellemek için
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,  # Daha stabil eğitim
    loftq_config=None,
)

In [ ]:
# ============================================
# 7. VERİ YÜKLEME
# ============================================
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# CSV'yi yükleyin
dataset_path = "/content/drive/My Drive/HukukPusulasi/combined3_reject_added.csv"
try:
    df = pd.read_csv(dataset_path)
    print("Dataset başarıyla yüklendi:")
    print(df.head())
    print(f"\nToplam satır sayısı: {len(df)}")
except FileNotFoundError:
    print(f"Hata: '{dataset_path}' dosyası bulunamadı.")
    raise
except Exception as e:
    print(f"Dataset yüklenirken hata: {e}")
    raise

In [ ]:
# ============================================
# QA PROMPT OLUŞTURMA
# ============================================
print("\n=== CONTEXT-AWARE PROMPT OLUŞTURULUYOR ===")

def create_conversation_list(row):
    """
    Question ve answer kullanarak ShareGPT formatında liste döndür.
    """
    question = row.get('question', '')
    answer = row.get('answer', '')

    # Direkt soru-cevap çifti
    return [
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": answer.strip()}
    ]

# 'conversations' sütununu oluştur
df['conversations'] = df.apply(create_conversation_list, axis=1)

print(f"✓ {len(df)} sohbet örneği oluşturuldu")
print("\nÖrnek sohbet:")
print("="*80)
example_conversation = df['conversations'].iloc[0]
print(f"Role: {example_conversation[0]['role']}\nContent: {example_conversation[0]['content']}...")
print(f"Role: {example_conversation[1]['role']}\nContent: {example_conversation[1]['content']}...")
print("="*80)

In [ ]:
# ============================================
# DATASET'İ HAZIRLA
# ============================================
print("\n=== DATASET HAZIRLANIYOR ===")

# Sadece 'conversations' sütununu al
dataset = Dataset.from_pandas(df[['conversations']])

print(f"Dataset boyutu: {len(dataset)}")
print(f"Columns: {dataset.column_names}")


In [ ]:
# ============================================
# 8. VERİ ÖN İŞLEME
# ============================================
from unsloth.chat_templates import standardize_sharegpt, apply_chat_template

print("\n=== VERİ ÖN İŞLEMESİ BAŞLIYOR ===")

# 'conversations' sütunu zaten ShareGPT formatında oluşturulduğu için 'to_sharegpt' adımı atlandı.

# Standardize et
dataset = standardize_sharegpt(dataset)

print("✓ Standardize edildi")
print(f"Dataset schema: {dataset.column_names}")

# Chat template uygula
dataset = apply_chat_template(
    dataset,
    tokenizer=tokenizer,
)

print("✓ Chat template uygulandı")
print(f"Final columns: {dataset.column_names}")


In [ ]:
# ============================================
# 9. VERİSETİNİ BÖLME
# ============================================
# Validation set (%5)
dataset_splits = dataset.train_test_split(test_size=0.1, seed=42)
temp_test = dataset_splits['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train': dataset_splits['train'],
    'validation': temp_test['train'],
    'test': temp_test['test']
})

print(f"Train: {len(dataset_dict['train'])}")
print(f"Validation: {len(dataset_dict['validation'])}")
print(f"Test: {len(dataset_dict['test'])}")

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from transformers.integrations import TensorBoardCallback
from transformers.trainer_callback import EarlyStoppingCallback

def is_bfloat16_supported():
    """Check if bfloat16 is supported"""
    return torch.cuda.get_device_properties(0).major >= 8

In [ ]:
# ============================================
# TRAINER ARGÜMANLARI
# ============================================
QUICK_TEST_MODE = False  # False yaparsan tam eğitim olur
MAX_STEPS_QUICK_TEST = 60  # 60 step'te durdur (hızlı test)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_dict['train'],
    eval_dataset=dataset_dict['validation'],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        # --- HIZLI TEST İÇİN BATCH AYARLARI ---
        per_device_train_batch_size=4 if QUICK_TEST_MODE else 2,      # 2'den 4'e (hızlanır)
        per_device_eval_batch_size=4 if QUICK_TEST_MODE else 2,       # 2'den 4'e
        gradient_accumulation_steps=2 if QUICK_TEST_MODE else 8,      # 8'den 2'ye (hızlanır)

        # --- HIZLI TEST: NUM_TRAIN_EPOCHS VE MAX_STEPS ---
        num_train_epochs=1 if not QUICK_TEST_MODE else 1,
        max_steps=MAX_STEPS_QUICK_TEST if QUICK_TEST_MODE else -1,

        # --- EVALUATION VE SAVE AYARLARI (HIZLI) ---
        eval_strategy="steps",
        eval_steps=30 if QUICK_TEST_MODE else 250,
        save_strategy="steps",
        save_steps=30 if QUICK_TEST_MODE else 250,
        logging_steps=10,

        # --- Optimizer ve LR ---
        learning_rate=2e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,

        # --- Model Kayıt (TEST İÇİN) ---
        save_total_limit=1 if QUICK_TEST_MODE else 2,
        load_best_model_at_end=False if QUICK_TEST_MODE else True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # --- Diğer ---
        report_to="none",  # "tensorboard" yerine "none" (hız)
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=3407,
        output_dir="outputs_quick_test" if QUICK_TEST_MODE else "outputs",  # Ayrı klasöre kaydet
    ),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1 if QUICK_TEST_MODE else 3,
            early_stopping_threshold=0.01
        )
    ]
)

print("✓ Trainer oluşturuldu")

In [ ]:
# ============================================
# 11. MEMORY STATS (EĞİTİM ÖNCESİ)
# ============================================
import subprocess
import re

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    output = result.stdout

    gpu_name_match = re.search(r'\|   \d  (.*)            Off', output)
    total_memory_match = re.search(r'\|    \d+MiB /  (\d+)MiB', output)
    used_memory_match = re.search(r'\|    (\d+)MiB /', output)

    if gpu_name_match and total_memory_match and used_memory_match:
        gpu_name = gpu_name_match.group(1).strip()
        total_memory_gb = int(total_memory_match.group(1)) / 1024
        used_memory_gb = int(used_memory_match.group(1)) / 1024

        print(f"\n=== GPU BİLGİSİ ===")
        print(f"GPU = {gpu_name}")
        print(f"Max memory = {total_memory_gb:.3f} GB")
        print(f"Kullanılan memory = {used_memory_gb:.3f} GB")
except Exception as e:
    print(f"GPU bilgisi alınamadı: {e}")

In [ ]:
# ============================================
# 12. MODELİ EĞİT
# ============================================
print("\n" + "="*80)
print("=== EĞİTİM BAŞLIYOR ===")
print("="*80 + "\n")

trainer_stats = trainer.train()

print("\n" + "="*80)
print("=== EĞİTİM TAMAMLANDI ===")
print("="*80)

In [ ]:
# ============================================
# 13. MEMORY STATS (EĞİTİM SONRASI)
# ============================================
import time

training_time_seconds = trainer.state.log_history[-1]['train_runtime']
training_time_minutes = training_time_seconds / 60

peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 * 1024)
peak_memory_gb = peak_memory_mib / 1024

print(f"\n=== EĞİTİM İSTATİSTİKLERİ ===")
print(f"Eğitim süresi: {training_time_minutes:.2f} dakika")
print(f"Peak memory kullanımı: {peak_memory_gb:.3f} GB")

In [ ]:
# ============================================
# 14. EĞİTİM SONRASI INFERENCE TESTİ
# ============================================
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "user", "content": "Kaza yaptım, sigorta şirketiyle nasıl ilerlemeliyim?"},
]

input_ids = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
print("\nEğitim sonrası cevap:")
_ = model.generate(input_ids, streamer=text_streamer, max_new_tokens=256,
                   pad_token_id=tokenizer.eos_token_id)

In [ ]:
# ============================================
# 15. TEST SETİ DEĞERLENDİRMESİ
# ============================================
import random
import tqdm
import traceback # Import traceback module

def evaluate_model(model, tokenizer, test_dataset, num_samples=5):
    """Test setinde model performansını değerlendir"""

    FastLanguageModel.for_inference(model)

    test_indices = random.sample(range(len(test_dataset)),
                                 min(num_samples, len(test_dataset)))

    results = []
    print("\n=== TEST SETİ DEĞERLENDİRMESİ BAŞLIYOR ===")
    for i, idx in enumerate(tqdm.tqdm(test_indices, desc="Test ediliyor")):
        sample = test_dataset[idx]

        question = ""
        expected_answer = ""
        try:
            # Check if 'conversations' key exists and is a list
            if 'conversations' in sample and isinstance(sample['conversations'], list) and len(sample['conversations']) >= 2:
                question = sample['conversations'][0].get('content', '').strip()
                expected_answer = sample['conversations'][1].get('content', '').strip()
            else:
                raise ValueError(f"'{idx}' indeksindeki sample['conversations'] beklenmedik formatta veya eksik: {sample.get('conversations', 'Yok')}")

            # Ensure question and expected_answer are not empty if they are critical
            if not question:
                 raise ValueError(f"'{idx}' indeksindeki soru boş.")
            if not expected_answer:
                 raise ValueError(f"'{idx}' indeksindeki beklenen cevap boş.")

        except (KeyError, IndexError, ValueError) as e:
            print(f"\n--- Hata: Örnek {idx} için conversation bilgisi alınamadı ---")
            print(f"Hata detayı: {e}")
            print(f"Problematic sample['conversations']: {sample.get('conversations', 'Anahtar bulunamadı')}")
            # Optional: print full traceback for deeper debugging
            # traceback.print_exc()
            continue

        messages = [{"role": "user", "content": question}]

        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        outputs = model.generate(
            input_ids,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Get only the newly generated tokens, excluding the input prompt
        generated_tokens = outputs[0][len(input_ids[0]):]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        results.append({
            'question': question,
            'generated': generated_text,
            'expected': expected_answer
        })

        print(f"\n--- Test {len(results)} ---")
        print(f"SORU: {question[:100]}...")
        print(f"MODEL: {generated_text[:200]}...")
        print(f"BEKLENEN: {expected_answer[:200]}...")

    if not results:
        print("\nHiçbir örnek başarıyla değerlendirilemedi. Lütfen yukarıdaki hata detaylarını kontrol edin.")
    return results

test_results = evaluate_model(model, tokenizer, dataset_dict['test'], num_samples=5)


In [ ]:
# ============================================
# 16. MODELİ KAYDETME
# ============================================
# HuggingFace Token'ı al
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None

# HuggingFace Hub'a yükle
if HF_TOKEN:
    print("HuggingFace Hub'a yükleniyor...")
    model.push_to_hub("beyzasn/hukuk-lora-llama-3-8b-v2", token=HF_TOKEN, private=True)
    tokenizer.push_to_hub("beyzasn/hukuk-lora-Llama-3-8b-v2", token=HF_TOKEN, private=True)
    print("✓ HuggingFace Hub'a yükleme tamamlandı")
else:
    print("⚠ HF_TOKEN bulunamadı, Hub'a yükleme yapılamadı")

In [ ]:
# ============================================
# 18. KAYDEDILMIŞ MODELİ YÜKLEME (İleride kullanmak için)
# ============================================

# HuggingFace Token'ı al
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None

# 2. LoRA adapterlerini yükle
if HF_TOKEN:
    model.load_adapter("beyzasn/hukuk-lora-llama-3-8b-v2", token=HF_TOKEN)  # HuggingFace Hub
    print("✓ Adapter başarıyla yüklendi.")
else:
    print("⚠ HF_TOKEN bulunamadı, adapter yüklenemedi.")


In [ ]:
# ============================================
# 17. GRADIO CHAT ARAYÜZÜ
# ============================================
# %%capture
# !pip install gradio

import gradio as gr
from transformers import TextIteratorStreamer, TextStreamer # TextIteratorStreamer eklendi
from threading import Thread # Thread eklendi

FastLanguageModel.for_inference(model)

def generate_response(message, history):
    """Gradio sohbet arayüzü için yanıt oluştur"""
    messages = []

    # Sohbet geçmişini ekle
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    # Güncel kullanıcı mesajını ekle
    messages.append({"role": "user", "content": message})

    # Girdi hazırla
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Gerçek zamanlı çıktı için streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Ayrı bir thread'de üretimi başlat
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Kısmi yanıtı sırayla gönder
    partial_response = ""
    for new_text in streamer:
        partial_response += new_text
        yield partial_response

# Örnek kullanıcı soruları
example_questions = [
    ["Piramit satış nedir? Bana basitçe anlatır mısınız?"],
    ["Kaza yaptım, sigorta şirketiyle nasıl ilerlemeliyim?"],
    ["Tüketici kredisi çekerken nelere dikkat etmeliyim?"],
    ["Online alışverişte cayma hakkım var mı? Ne kadar sürem var?"],
    ["Aldığım üründe ayıp varsa ne yapmalıyım? Haklarım nelerdir?"]
]

# Gradio arayüzü oluştur
demo = gr.ChatInterface(
    fn=generate_response,
    examples=example_questions,
    title="⚖️ Hukuk Pusulası - AI Hukuki Asistan",
    description="""
    Tüketici hukuku konularında sorularınızı sorabilirsiniz.
    Aşağıdaki örnek sorulardan birini seçebilir veya kendi sorunuzu yazabilirsiniz.

    **Not:** Bu bir AI modelidir ve verdiği bilgiler sadece bilgilendirme amaçlıdır.
    Kesin hukuki bilgi için bir avukata danışmanız önerilir.

    ### 💡 İpuçları:
    - Sorularınızı açık ve net bir şekilde sorun
    - Birden fazla soru sorabilirsiniz (chat formatında)
    - Daha detaylı bilgi için ek sorular sorabilirsiniz
    """,
    theme="soft",
    submit_btn="📤 Gönder",
)

# Arayüzü başlat
print("\n=== GRADIO ARAYÜZÜ BAŞLATILIYOR ===")
demo.launch(share=True, debug=True)
